# LiteMIL Usage Examples

This notebook demonstrates the complete LiteMIL pipeline:
1. **Feature extraction** from raw WSI files
2. **Training** with nested cross-validation
3. **Inference** on pre-extracted features
4. **Attention visualization** for model interpretability

---

## 📦 Setup and Configuration

In [ ]:
# Core imports
from MILS.LiteMIL import LiteMIL
from MILS.MABMIL import ABMIL, ABMIL_Multihead
from MILS.TransMIL import TransMIL
from MILS.pool import meanPool, maxPool

import torch
import numpy as np
import h5py
from pathlib import Path

In [ ]:
# Configuration
DATASET = 'breast'  # Options: 'breast', 'lung', 'kidney', 'tupac'
MIL_MODEL = 'LiteMIL'  # Options: 'LiteMIL', 'ABMIL', 'MADMIL', 'TransMIL', 'maxPool', 'meanPool'

# Paths
FEATURES_DIR = f'datasets/{DATASET}/features'
LABELS_CSV = f'datasets/{DATASET}/labels.csv'
OUTPUT_DIR = f'outputs/{MIL_MODEL}/{DATASET}'

# Dataset configuration
CLASS_DICT = {
    'breast': ['IDC', 'ILC'],
    'lung': ['LUAD', 'LUSC'],
    'kidney': ['PRCC', 'CCRCC', 'CHRCC'],
    'tupac': ['Low', 'High']
}
CLASS_NAMES = CLASS_DICT[DATASET]
NUM_CLASSES = len(CLASS_NAMES)

print(f"Dataset: {DATASET}")
print(f"Classes: {CLASS_NAMES}")
print(f"Model: {MIL_MODEL}")
print(f"Output directory: {OUTPUT_DIR}")

---
## 🔬 1. Feature Extraction from Raw WSIs

Extract features from raw whole slide images (.svs, .tif, etc.).

In [ ]:
from utils.feature_extractor import WSIFeatureExtractor

# Initialize feature extractor
extractor = WSIFeatureExtractor(
    backbone='resnet50',  # Options: 'resnet50', 'phikon-v2', 'uni'
    patch_size=256,
    stride=256,
    level=0,              # 0 = highest magnification
    tissue_threshold=0.5,
    batch_size=64,
    device='cuda'
)

print("✓ Feature extractor initialized")

### 1.1 Extract features from a single WSI


In [ ]:
# Extract features from raw WSI
WSI_PATH = 'testSlides/testSlide.svs'  # Update with your WSI path
OUTPUT_FEATURES = 'testSlides/testSlide_extracted.h5'

print(f"Extracting features from: {WSI_PATH}")
print("This may take several minutes depending on slide size...\n")

extraction_result = extractor.process(
    wsi_path=WSI_PATH,
    save_path=OUTPUT_FEATURES
)

print("\n" + "="*70)
print("FEATURE EXTRACTION RESULTS")
print("="*70)
print(f"Extracted patches: {len(extraction_result['features'])}")
print(f"Feature shape: {extraction_result['features'].shape}")
print(f"Coordinates shape: {extraction_result['coords'].shape}")
print(f"Tissue ratio: {extraction_result['metadata']['tissue_ratio']:.2%}")
print(f"Saved to: {OUTPUT_FEATURES}")
print("="*70)

### 1.2 Extract features from the entire WSIs dataset
Follow the instructions in README.md for command line feature extraction (Batch processing)

In [ ]:
# Batch processing
#!python extractFeatures.py --input_dir datasets/breast/wsi/ --output_dir datasets/breast/features/ --backbone phikon-v2 --patch_size 256 --level 0 --format h5

---
## 🎯 2. Training with Nested Cross-Validation

Train the model with nested cross-validation for unbiased evaluation.

In [ ]:
# Model configuration
MODEL_CONFIG = {
    'LiteMIL': LiteMIL(
        input_dim=1024,
        hidden_dim=256,
        num_classes=NUM_CLASSES,
        num_heads=4,
        num_queries=1,
        dropout=0.25
    ),
    'ABMIL': ABMIL(num_classes=NUM_CLASSES),
    'MADMIL': ABMIL_Multihead(num_classes=NUM_CLASSES),
    'TransMIL': TransMIL(input_size=1024, num_classes=NUM_CLASSES),
    'maxPool': maxPool(input_dim=1024, hidden_dim=256, dropout=0.25, num_classes=NUM_CLASSES),
    'meanPool': meanPool(input_dim=1024, hidden_dim=256, dropout=0.25, num_classes=NUM_CLASSES)
}

# Model factory
model_factory = lambda: MODEL_CONFIG[MIL_MODEL]

print(f"✓ Model factory created for {MIL_MODEL}")

In [ ]:
from utils.mil_dataset import MILDataset
from utils.nested_cv import NestedCrossValidation

# Load dataset
print("Loading dataset...")
dataset = MILDataset(
    csv_path=LABELS_CSV,
    features_dir=FEATURES_DIR,
    class_names=CLASS_NAMES,
    mode='chunked',  # or 'full'
    chunk_size=1000
)

print(f"✓ Loaded {len(dataset)} samples")

# Setup nested cross-validation
cv = NestedCrossValidation(
    model_factory=model_factory,
    dataset=dataset,
    class_names=CLASS_NAMES,
    output_dir=OUTPUT_DIR,
    n_outer=2,  # Use 5 for full evaluation
    n_inner=2,  # Use 4 for full evaluation
    batch_size=16,
    epochs=1,   # Use 100 for full training
    patience=10,
    device='cuda',
    use_amp=True
)

print("\nStarting nested cross-validation...")
print("Note: Using reduced folds/epochs for demonstration. Increase for production.")

In [ ]:
# Run training
results = cv.run()

# Display results
print("\n" + "="*70)
print("TRAINING RESULTS")
print("="*70)
print(f"Chunk-level Accuracy: {results['chunk_accuracy_mean']:.3f} ± {results['chunk_accuracy_std']:.3f}")
print(f"Slide-level Accuracy: {results['slide_accuracy_mean']:.3f} ± {results['slide_accuracy_std']:.3f}")
print(f"Best Overall Accuracy: {results['best_overall_accuracy']:.3f}")
print(f"\nModels saved to: {OUTPUT_DIR}")
print("="*70)

---
## 🔮 3. Inference
### 3.1 Inference on Pre-extracted Features

In [ ]:
from utils.inference import SlidePredictor

# Create predictor with trained model
predictor = SlidePredictor(
    model_path=f'{OUTPUT_DIR}/all_folds_best.pth',
    model_class=model_factory,
    class_names=CLASS_NAMES,
    device='cuda'
)

print("✓ Predictor loaded")

In [ ]:
# Example 1: Full mode inference
TEST_SLIDE = 'testSlides/testSlide.h5'  # Update with your test slide path

print(f"Running inference on: {TEST_SLIDE}")
result = predictor.predict(TEST_SLIDE, mode='full')

print("\n" + "="*70)
print("PREDICTION RESULTS (Full Mode)")
print("="*70)
print(f"Predicted Class: {result['predicted_class']}")
print(f"Confidence: {result['confidence']:.2%}")
print(f"\nClass Probabilities:")
for cls, prob in result['probabilities'].items():
    bar = '█' * int(prob * 40)
    print(f"  {cls:10s}: {prob:.3f} {bar}")
print(f"\nNum instances: {result['num_instances']}")
print("="*70)

In [ ]:
# Example 2: Chunked mode inference (for large slides)
result_chunked = predictor.predict(TEST_SLIDE, mode='chunked', chunk_size=1000)

print("\n" + "="*70)
print("PREDICTION RESULTS (Chunked Mode)")
print("="*70)
print(f"Predicted Class: {result_chunked['predicted_class']}")
print(f"Confidence: {result_chunked['confidence']:.2%}")
print(f"\nNum chunks: {result_chunked['num_chunks']}")
print(f"Total instances: {result_chunked['num_instances']}")

if 'chunk_attention_stats' in result_chunked:
    stats = result_chunked['chunk_attention_stats']
    print(f"\nChunk Attention Statistics:")
    print(f"  Min: {stats['min']:.3f}")
    print(f"  Max: {stats['max']:.3f}")
    print(f"  Mean: {stats['mean']:.3f}")
    print(f"  Std: {stats['std']:.3f}")
print("="*70)

### 3.2 Direct Inference on Raw WSI

You can also predict directly from raw WSI files (combines extraction + inference).

In [ ]:
from utils.inference import extractPredict

# Create predictor with on-the-fly feature extraction
wsi_predictor = extractPredict(
    model_path=f'{OUTPUT_DIR}/all_folds_best.pth',
    model_class=model_factory,
    class_names=CLASS_NAMES,
    backbone='resnet50',
    patch_size=256,
    stride=256,
    level=0,
    device='cuda'
)

print("✓ WSI predictor initialized")
print("Running inference directly from raw WSI...\n")

wsi_result = wsi_predictor.predict(WSI_PATH, mode='chunked')

print("\n" + "="*70)
print("DIRECT WSI PREDICTION")
print("="*70)
print(f"Predicted: {wsi_result['predicted_class']}")
print(f"Confidence: {wsi_result['confidence']:.2%}")
print("="*70)

---
## 📊 4. Attention Visualization

Generate interpretable attention heatmaps to understand model predictions.

In [ ]:
from utils.attention_visualizer import AttentionVisualizer
import matplotlib.pyplot as plt

# Configuration for visualization
FEATURES_WITH_COORDS = 'testSlides/testSlide.h5'  # Must have coordinates
WSI_FOR_VIZ = 'testSlides/testSlide.svs'  # Original WSI file

# Create output directory
VIZ_OUTPUT = Path('outputs/visualizations/')
VIZ_OUTPUT.mkdir(parents=True, exist_ok=True)

print("✓ Visualization setup complete")

### 4.1 Basic Attention Heatmap

In [ ]:
# Run inference and get attention
result = predictor.predict(FEATURES_WITH_COORDS, mode='full')

# Load coordinates
with h5py.File(FEATURES_WITH_COORDS, 'r') as f:
    if 'coords' not in f:
        print("❌ Error: No coordinates found in feature file!")
        print("Please re-extract features with coordinates using WSIFeatureExtractor.")
    else:
        coords = f['coords'][:]
        print(f"✓ Loaded {len(coords)} patch coordinates")

# Get attention scores
attention = result['patch_attention'].numpy()

print(f"\nAttention Statistics:")
print(f"  Shape: {attention.shape}")
print(f"  Min: {attention.min():.4f}")
print(f"  Max: {attention.max():.4f}")
print(f"  Mean: {attention.mean():.4f}")
print(f"  Std: {attention.std():.4f}")

In [ ]:
# Create visualizer
visualizer = AttentionVisualizer(cmap='jet', alpha=0.5)

# Generate simple heatmap
fig = visualizer.visualize_full_attention(
    WSI_FOR_VIZ,
    coords,
    attention,
    patch_size=256,
    level=0,
    top_k=10,
    save_path=VIZ_OUTPUT / 'simple_heatmap.png'
)

plt.show()
print(f"✓ Saved to: {VIZ_OUTPUT / 'simple_heatmap.png'}")

### 4.2 Detailed Visualization with Statistics

In [ ]:
# Generate comprehensive visualization
fig_detailed = visualizer.visualize_detailed_attention(
    WSI_FOR_VIZ,
    coords,
    attention,
    result,
    save_path=VIZ_OUTPUT / 'detailed_visualization.png',
    patch_size=256,
    level=0,
    top_k=10
)

plt.show()
print(f"✓ Saved to: {VIZ_OUTPUT / 'detailed_visualization.png'}")

### 4.3 Extract Top Attended Patches

In [ ]:
# Extract top-10 patches with highest attention
patches, top_indices = visualizer.extract_top_patches(
    WSI_FOR_VIZ,
    coords,
    attention,
    top_k=10,
    patch_size=256,
    level=0,
    save_dir=VIZ_OUTPUT / 'top_patches/'
)

print(f"\nTop-10 Patch Indices: {top_indices}")
print(f"Top-10 Attention Scores:")
for i, (idx, score) in enumerate(zip(top_indices, attention[top_indices]), 1):
    print(f"  Rank {i}: Index {idx}, Score {score:.4f}")
print(f"Top 10 Patches Saved to: {VIZ_OUTPUT / 'top_patches/'}" )

In [ ]:
# Display patches in grid
fig_grid = visualizer.create_patch_grid(
    patches,
    attention[top_indices],
    save_path=VIZ_OUTPUT / 'patch_grid.png',
    cols=5
)

plt.show()
print(f"✓ Saved to: {VIZ_OUTPUT / 'patch_grid.png'}")

### 4.4 Compare Different Colormaps

In [ ]:
import openslide

colormaps = ['jet', 'hot', 'viridis', 'plasma']

fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes = axes.flatten()

slide = openslide.OpenSlide(WSI_FOR_VIZ)
thumbnail = slide.get_thumbnail((2048, 2048))
thumbnail_array = np.array(thumbnail)

for ax, cmap in zip(axes, colormaps):
    vis = AttentionVisualizer(cmap=cmap, alpha=0.5)
    
    # Create heatmap
    heatmap = vis._create_heatmap(
        coords,
        vis._normalize_attention(attention),
        slide.dimensions,
        256, 0, thumbnail_array.shape[:2]
    )
    
    # Display
    ax.imshow(thumbnail)
    heatmap_overlay = vis.cmap(heatmap)
    heatmap_overlay[..., 3] = heatmap * vis.alpha
    ax.imshow(heatmap_overlay)
    ax.set_title(f'Colormap: {cmap}', fontsize=14, fontweight='bold')
    ax.axis('off')

slide.close()

plt.tight_layout()
plt.savefig(VIZ_OUTPUT / 'colormap_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved to: {VIZ_OUTPUT / 'colormap_comparison.png'}")

### 4.5 Attention Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Histogram
axes[0].hist(attention, bins=50, color='steelblue', edgecolor='black')
axes[0].axvline(attention.mean(), color='red', linestyle='--', label=f'Mean: {attention.mean():.3f}')
axes[0].axvline(np.median(attention), color='green', linestyle='--', label=f'Median: {np.median(attention):.3f}')
axes[0].set_xlabel('Attention Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Attention Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Box plot
axes[1].boxplot(attention, vert=True)
axes[1].set_ylabel('Attention Score')
axes[1].set_title('Attention Box Plot')
axes[1].grid(True, alpha=0.3)

# 3. Cumulative distribution
sorted_attn = np.sort(attention)
cumulative = np.arange(1, len(sorted_attn) + 1) / len(sorted_attn)
axes[2].plot(sorted_attn, cumulative, linewidth=2, color='steelblue')
axes[2].axhline(0.9, color='red', linestyle='--', label='90th percentile')
axes[2].axhline(0.95, color='orange', linestyle='--', label='95th percentile')
axes[2].set_xlabel('Attention Score')
axes[2].set_ylabel('Cumulative Probability')
axes[2].set_title('Cumulative Distribution')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(VIZ_OUTPUT / 'attention_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Print detailed statistics
print("\n" + "="*70)
print("ATTENTION STATISTICS")
print("="*70)
print(f"Min: {attention.min():.4f}")
print(f"Max: {attention.max():.4f}")
print(f"Mean: {attention.mean():.4f}")
print(f"  Median: {np.median(attention):.4f}")
print(f"  Std: {attention.std():.4f}")
print(f"  90th percentile: {np.percentile(attention, 90):.4f}")
print(f"  95th percentile: {np.percentile(attention, 95):.4f}")
print(f"  99th percentile: {np.percentile(attention, 99):.4f}")

### 4.6 Chunked Inference Visualization

In [ ]:
# Run chunked inference
result_chunked = predictor.predict(FEATURES_WITH_COORDS, mode='chunked', chunk_size=100)

print(f"Predicted: {result_chunked['predicted_class']}")
print(f"Confidence: {result_chunked['confidence']:.2%}")
print(f"\nChunk attention stats: {result_chunked['chunk_attention_stats']}")

In [ ]:
# Visualize chunk-level attention
fig_chunked = visualizer.visualize_chunked_attention(
    WSI_FOR_VIZ,
    coords,
    result_chunked['chunk_attention'],
    chunk_size=1000,
    save_path=VIZ_OUTPUT /'chunked_attention.png',
    patch_size=256,
    level=0
)

plt.show()

### 4.7 Save Attention for Further Analysis

In [ ]:
# Save attention data
np.savez(
    VIZ_OUTPUT / 'attention_data.npz',
    attention=attention,
    coords=coords,
    prediction=result['predicted_class'],
    confidence=result['confidence'],
    probabilities=list(result['probabilities'].values())
)

print("✓ Attention data saved to attention_data.npz")

In [ ]:
# Load and verify
data = np.load(VIZ_OUTPUT / 'attention_data.npz', allow_pickle=True)

print("Saved data keys:", list(data.keys()))
print(f"\nAttention shape: {data['attention'].shape}")
print(f"Coords shape: {data['coords'].shape}")
print(f"Prediction: {data['prediction']}")
print(f"Confidence: {data['confidence']:.2%}")

## 📄 Citation

If you use LiteMIL pipeline in your research, please cite:

```bibtex
@article{kussaibi2025litemil,
  title={LiteMIL: A Computationally Efficient Cross-Attention MIL for Cancer Subtyping on WSIs},
  author={Kussaibi, Haitham},
  journal={Journal of Medical Imaging},
  year={2025}
}
```

---

## 📜 License

This project is licensed under the Apache 2.0 License - see the [LICENSE](LICENSE) file for details.

## 📧 Corresponding Author

**Dr. Haitham Kussaibi**
- 📧 Email: kussaibi@gmail.com
- 🔗 ORCID: [0000-0002-9570-0768](https://orcid.org/0000-0002-9570-0768)
- 💼 LinkedIn: [linkedin.com/in/haithamkussaibi](https://www.linkedin.com/in/haithamkussaibi/)

---

## ⭐ Star History

If you find LiteMIL useful for your research, please consider giving it a star! ⭐

---

**Made with ❤️ for the computational pathology community**